In [1]:
 
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

Transaction data
    Transaction id: Id related to the transaction 
    TransactionDT: timedelta from a given reference datetime (not an actual timestamp)
    TransactionAMT: transaction payment amount in USD
    ProductCD [Categorical]: product code(the product for each transaction)
    card1–6 [Categorical]: payment card related information like card type, country etc
    addr1, addr2 [Categorical]: address information
    dist1,dist2: some distance information
    P_emaildomain [Categorical]: email domain of purchaser.
    R_emaildomain [Categorical]: email domain of the recipient.
    C1-C14: counting, such as how many addresses are found to be associated with the payment card, etc. The actual meaning is masked.
    D1-D15: time delta, such as days between the previous transactions, etc.
    M1-M9 [Categorical]: match, such as names on card and address, etc.
    Vxxx: Vesta engineered rich features, including ranking, counting, and other entity relations.
    Identity data

Transaction id: Id related to the transaction
    DeviceType [Categorical]: Type of device used for the transaction
    DeviceInfo [Categorical]: More information about device used
    id 1–38 [Categorical+numeric]: network connection information,browser information etc (id 12–38 are categorical information)

In [2]:
try:
    train_transactions = pd.read_csv('../data/ieee-fraud-detection/train_transaction.csv')
    train_identity = pd.read_csv('../data/ieee-fraud-detection/train_identity.csv')
    print(f"✓ Loaded transactions: {train_transactions.shape}")
    print(f"✓ Loaded identity: {train_identity.shape}")

    test_transactions = pd.read_csv('../data/ieee-fraud-detection/test_transaction.csv')
    test_identity = pd.read_csv('../data/ieee-fraud-detection/test_identity.csv')
    print(f"✓ Loaded transactions: {test_transactions.shape}")
    print(f"✓ Loaded identity: {test_identity.shape}")
except FileNotFoundError:
    print("⚠ Data files not found. Download from Kaggle:")
    raise

✓ Loaded transactions: (590540, 394)
✓ Loaded identity: (144233, 41)
✓ Loaded transactions: (506691, 393)
✓ Loaded identity: (141907, 41)


In [3]:
# Merge on TransactionID

def merge_data(data_type, transactions, identity):
    print(f"\nMerging {data_type} data...")
    data = transactions.merge(identity, on='TransactionID', how='left')
    print(f"✓ Merged shape: {data.shape}")

    if data_type == "train":
        print(f"✓ Fraud rate: {data['isFraud'].mean():.3%}")
    print(f"✓ Missing values:\n{data.isnull().sum()[data.isnull().sum() > 0].head()}")

    return data

data = merge_data("train", train_transactions, train_identity)
data_test = merge_data("test", test_transactions, test_identity)
 




Merging train data...
✓ Merged shape: (590540, 434)
✓ Fraud rate: 3.499%
✓ Missing values:
card2    8933
card3    1565
card4    1577
card5    4259
card6    1571
dtype: int64

Merging test data...
✓ Merged shape: (506691, 433)
✓ Missing values:
card2    8654
card3    3002
card4    3086
card5    4547
card6    3007
dtype: int64


In [4]:
print("\n[3] Understanding temporal structure...")
 
# TransactionDT is seconds from first transaction
print(f"✓ TransactionDT range:")
print(f"    Min: {data['TransactionDT'].min():,} seconds")
print(f"    Max: {data['TransactionDT'].max():,} seconds")
 
# Convert to days
data['days_from_start'] = data['TransactionDT'] / (3600 * 24)
 
total_days = data['days_from_start'].max()
total_months = total_days / 30
 
print(f"✓ Time span: {total_days:.0f} days (~{total_months:.0f} months)")
 
# Check transactions per day
txns_per_day = data.groupby(data['days_from_start'].astype(int)).size()
print(f"✓ Transactions per day: {txns_per_day.mean():.0f} avg, {txns_per_day.max():.0f} max")


[3] Understanding temporal structure...
✓ TransactionDT range:
    Min: 86,400 seconds
    Max: 15,811,131 seconds
✓ Time span: 183 days (~6 months)
✓ Transactions per day: 3245 avg, 6852 max


In [5]:
print("\n[4] Creating temporal train/val/test split from TRAIN data...")
 
# Split by time quantiles to get roughly equal-sized splits
# Assume ~180 days of data (6 months)
 
train_cutoff = total_days * (2/3)      # First 2/3 → train
val_cutoff = total_days * (5/6)        # Next 1/6 → val
# Remaining 1/6 → test
 
train_mask = data['days_from_start'] < train_cutoff
val_mask = (data['days_from_start'] >= train_cutoff) & (data['days_from_start'] < val_cutoff)
test_mask = data['days_from_start'] >= val_cutoff
 
train_data = data[train_mask].copy()
val_data = data[val_mask].copy()
test_data = data[test_mask].copy()
 
print(f"\n✓ Temporal split (from TRAIN set only):")
print(f"  Train: {len(train_data):,} rows ({train_data['isFraud'].mean():.3%} fraud)")
print(f"  Val:   {len(val_data):,} rows ({val_data['isFraud'].mean():.3%} fraud)")
print(f"  Test:  {len(test_data):,} rows ({test_data['isFraud'].mean():.3%} fraud)")
 
# Verify no time overlap
print(f"\n✓ Temporal ordering check:")
print(f"  Train: days {train_data['days_from_start'].min():.0f} - {train_data['days_from_start'].max():.0f}")
print(f"  Val:   days {val_data['days_from_start'].min():.0f} - {val_data['days_from_start'].max():.0f}")
print(f"  Test:  days {test_data['days_from_start'].min():.0f} - {test_data['days_from_start'].max():.0f}")
 
# Verify no overlap
assert train_data['days_from_start'].max() <= val_data['days_from_start'].min(), \
    "Train/Val overlap!"
assert val_data['days_from_start'].max() <= test_data['days_from_start'].min(), \
    "Val/Test overlap!"
print(f"  ✓ No time overlap (correct!)")


[4] Creating temporal train/val/test split from TRAIN data...

✓ Temporal split (from TRAIN set only):
  Train: 417,554 rows (3.526% fraud)
  Val:   84,710 rows (3.392% fraud)
  Test:  88,276 rows (3.477% fraud)

✓ Temporal ordering check:
  Train: days 1 - 122
  Val:   days 122 - 152
  Test:  days 152 - 183
  ✓ No time overlap (correct!)


In [6]:
 
print("\n[5] Creating prepared dataset...")
 
prepared_data = {
    'train': train_data,
    'val': val_data,
    'test': test_data,
    'full': data,
    'splits': {
        'train_cutoff': train_cutoff,
        'val_cutoff': val_cutoff
    },
    'metadata': {
        'source': 'IEEE-CIS Fraud Detection Competition (train set only)',
        'total_rows': len(data),
        'total_fraud': data['isFraud'].sum(),
        'fraud_rate': data['isFraud'].mean(),
        'total_days': total_days
    }
}


[5] Creating prepared dataset...


In [7]:
 
print("\n[6] Saving prepared data...")
 
output_path = '../data/ieee_prepared.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(prepared_data, f)
 
file_size = Path(output_path).stat().st_size / 1024 / 1024
print(f"✓ Saved to {output_path} ({file_size:.1f} MB)")


[6] Saving prepared data...
✓ Saved to ../data/ieee_prepared.pkl (3907.0 MB)
